# Project 3 - Are Forecasters Rational?

In [292]:
import pandas as pd
import numpy as np
rng = np.random.default_rng()

The above code is just importing NumPy and Pandas, while also creating a random number generator

In [294]:
prunemp = pd.read_excel("data/SPFmicrodata.xlsx", sheet_name='PRUNEMP')
prunemp.set_index(['YEAR'],inplace=True)

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [295]:
prunemp_short = prunemp.iloc[:,13:23]
prunemp_short = prunemp_short.div(prunemp_short.sum(axis=1), axis=0)
prunemp_short['100%']= prunemp_short.sum(axis=1)
prunemp_short['ID']=prunemp['ID']
prunemp_short['QUARTER']=prunemp['QUARTER']
prunemp_short = prunemp_short.loc['2009':'2025']
prunemp_short.reset_index(inplace=True)
prunemp_short['YEAR1'] = prunemp_short['YEAR']
prunemp_short['QUARTER1']=prunemp_short['QUARTER']
prunemp_short['forecast_year']=prunemp_short['YEAR1']+1
prunemp_short.set_index(['ID','YEAR','QUARTER'], inplace=True)
prunemp_short = prunemp_short[prunemp_short['100%']!=0]

In [296]:
prunemp_short.head()

,,,PRUNEMP11,PRUNEMP12,PRUNEMP13,PRUNEMP14,PRUNEMP15,PRUNEMP16,PRUNEMP17,PRUNEMP18,PRUNEMP19,PRUNEMP20,100%,YEAR1,QUARTER1,forecast_year
ID,YEAR,QUARTER,,,,,,,,,,,,,,
20,2009,2,0.00,0.00,0.20,0.60,0.20,0.00,0.0,0.0,0.0,0.0,1.0,2009,2,2010
84,2009,2,0.20,0.65,0.10,0.05,0.00,0.00,0.0,0.0,0.0,0.0,1.0,2009,2,2010
407,2009,2,0.00,0.00,0.05,0.70,0.15,0.10,0.0,0.0,0.0,0.0,1.0,2009,2,2010
411,2009,2,0.05,0.15,0.25,0.30,0.20,0.05,0.0,0.0,0.0,0.0,1.0,2009,2,2010
420,2009,2,0.00,0.00,0.35,0.55,0.10,0.00,0.0,0.0,0.0,0.0,1.0,2009,2,2010


### Importing and Remodeling PRUNEMP spreadsheet:

1. First, the excel file was imported using the pd.read_excel method to create a df, and called `prunemp`.
2. The index for prunemp was set to the 'YEAR' column.
3. `prunemp_short` is created. This is the df that will be used for the forecaster predictions, for the remainder of this project. prunemp_short is prunemp, but ONLY PRUNEMP11-20, as those are the required columns for forecaster predictions of next year's unemployment.
4. The next line divides each value in each row by its rows sum. For most rows this will just give us the exact values as percents (if the total=100), but for the rows that are are wrong/don't equal to 100, this will make it so that the row does equal to 100%, fixing the data.
5. The addition of the 100% column just makes sure that all values equal to 100% or 1.0, adding all of the probabilities.
6. ID and QUARTER columns are added, they will be made into a multi-index later in the code.
7. The data is reduced again just to the years 2009 and on, since that is the data we are looking at.
8. The indices are reset (just so that we can make the multi-index)
9. YEAR1 and QUARTER1 are created, these are the exact same columns as YEAR and QUARTER, but they exist just to be used when sampling later on, as YEAR and QUARTER will soon be indices.
10. forecast_year denotes the year that the row applies to. Since they are predictions for next year, its just year + 1.
11. Indices are set to YEAR, QUARTER, and ID - giving each row a distinct value.
12. Last, we only include rows where 100% is actually = 100% or 1.0. This removes all of the rows where 100% is equal to 0, removing the remaining NaN rows, where data is simply not present and therefore gives us 0. (ex - year is 2009 but quarter is 1.)

The head of prunemp_short depicts all of these aspects of the dataframe.

In [298]:
real_unemp = pd.read_excel("data/UNRATE.xlsx", sheet_name = 'Monthly', )
real_unemp['observation_date']=pd.to_datetime(real_unemp['observation_date'])
real_unemp.set_index('observation_date', inplace=True)
real_unemp = real_unemp.resample('YE').mean()
real_unemp = real_unemp[pd.to_datetime('2009-01-01'):]
real_unemp['forecast_year']=list(range(2009,2026))

In [299]:
real_unemp

,UNRATE,forecast_year
observation_date,,
2009-12-31,9.283333,2009
2010-12-31,9.608333,2010
2011-12-31,8.933333,2011
2012-12-31,8.075000,2012
2013-12-31,7.358333,2013
2014-12-31,6.158333,2014
2015-12-31,5.275000,2015
2016-12-31,4.875000,2016
2017-12-31,4.358333,2017


### Importing and Remodeling UNRATE spreadsheet:

1. First, the excel file was imported using the pd.read_excel method to create a df, and called real_unemp.
2. Index was changed to observation_date, which is date_time, as we are given exact dates. This makes the data look better, and allows for indexing based on dates.
3. We then resample and take the mean for ALL of the data by __year__, in order to get the unemployment rate for each year, not just each day
4. Then, we just take the years 2009 and on, matching the data from the previous DF.
5. Finally, we add a forecast_year to match the previous dataframe.

The data is shown by calling real_unemp

In [301]:
def get_ints(year, quarter):
    if year > 2024 or (year == 2024 and quarter >= 2):
        return [(0,3.1),(3.1,3.6),(3.7,4.2),(4.3,4.8),(4.9,5.4),(5.5,6.0),(6.1,7.1),(7.2, 8.2),(8.3,9.8),(9.9,100)]
    elif year > 2020 or (year==2020 and (quarter>=2)):
        return [(0,2.9),(3.0,3.9),(4.0,4.9),(5.0,5.9),(6.0,6.9),(7.0,7.9),(8.0,9.9),(10.0,11.9),(12.0,14.9),(15.0,100)]
    elif year >= 2014:
        return [(0,3.9),(4.0,4.9),(5.0,5.4),(5.5,5.9),(6.0,6.4),(6.5,6.9),(7.0,7.4),(7.5,7.9),(8.0,8.9),(9.0,100)]
    else:
        return [(0,5.9),(6.0,6.9),(7.0,7.4),(7.5,7.9),(8.0,8.4),(8.5,8.9),(9.0,9.4),(9.5,9.9),(10.0,10.9),(11.0,100)]

### `get_ints`

This method assigns the correct intervals to each forecasting row from the documentation of the PRUNEMP spreadsheet. Based on a row's year & quarter, the row is assigned certain intervals which the probabilities will correlate to. This will be used in sample_func2 below.

In [303]:
def sample_func2(row):
    N=1000
    intervals = get_ints(row['YEAR1'],row['QUARTER1'])
    probs = [row[f'PRUNEMP{i}'] for i in range(11, 21)]
    interval_idx = rng.choice(len(intervals),size=N, p=probs)
    samples = []
    for i in interval_idx:
        low = intervals[int(i)][0]
        high = intervals[int(i)][1]
        samples.append(rng.uniform(low, high))
    return np.array(samples)

In [304]:
prunemp_short['sample_values'] = prunemp_short.apply(sample_func2, axis=1)

### `sample_func2` method:

The function is meant to take the probabilities and intervals for each row of a dataframe, and based on their contents they will create a sample of 1000 "predictions." Input is a row. It works like this:

1. Sets N = 1000 (for iterations of creating samples)
2. intervals = the list of tuples from the `get_ints` method previously mentioned. Gets the correct list using the YEAR1 and QUARTER1 from the row in prunemp_short.
3. probs = a list of all of the values from PRUNEMP11 to PRUNEMP20. These are the probabilities that the rate will be within the intervals.
4. interval_idx randomly selects N intervals from intervals, with each chosen based on it's associated probability in probs.
5. For each interval in interval_idx:
   1. low = the interval in intervals 1st value
   2. high = the interval in intervals 2nd value. Both are set to be integers because as of now they are dtype = int64.
   3. the samples list is appended to take a random uniform number in between the low and high value. This is a sampled predicition that the forecaster may choose.
6. Return the array of samples.

Then, I set a column within prunemp_short, labeled `sample_values` to be the outcome of sample_func2, for all rows (axis=1). This can be seen below when I call prunemp_short.head() once again.

In [306]:
def confidence_intervals(sample_values):
    lower_ci = np.percentile(sample_values, 2.5)
    upper_ci = np.percentile(sample_values, 97.5)
    return pd.Series({'lower_CI': lower_ci, 'upper_CI': upper_ci})

ci_prunemp = prunemp_short['sample_values'].apply(confidence_intervals)
prunemp_short = prunemp_short.join(ci_prunemp)

In [307]:
prunemp_short = prunemp_short.reset_index()
prunemp_short = prunemp_short.merge(real_unemp, on='forecast_year', how='left')
prunemp_short.drop(['YEAR1','QUARTER1'],axis=1, inplace=True)
prunemp_short.set_index(['ID','YEAR','QUARTER'], inplace=True)

In [308]:
prunemp_short['in_CI'] = ((prunemp_short['UNRATE'] >= prunemp_short['lower_CI']) & (prunemp_short['UNRATE'] <= prunemp_short['upper_CI']))

In [309]:
prunemp_short.head()

,,,PRUNEMP11,PRUNEMP12,PRUNEMP13,PRUNEMP14,PRUNEMP15,PRUNEMP16,PRUNEMP17,PRUNEMP18,PRUNEMP19,PRUNEMP20,100%,forecast_year,sample_values,lower_CI,upper_CI,UNRATE,in_CI
ID,YEAR,QUARTER,,,,,,,,,,,,,,,,,
20,2009,2,0.00,0.00,0.20,0.60,0.20,0.00,0.0,0.0,0.0,0.0,1.0,2010,"[7.65422228235436, 8.050195094037713, 7.784252...",7.048328,8.349712,9.608333,False
84,2009,2,0.20,0.65,0.10,0.05,0.00,0.00,0.0,0.0,0.0,0.0,1.0,2010,"[6.763998144730916, 6.622121999639133, 6.75377...",0.902951,7.720981,9.608333,False
407,2009,2,0.00,0.00,0.05,0.70,0.15,0.10,0.0,0.0,0.0,0.0,1.0,2010,"[7.610279175889179, 8.506460734663333, 8.05742...",7.154574,8.795445,9.608333,False
411,2009,2,0.05,0.15,0.25,0.30,0.20,0.05,0.0,0.0,0.0,0.0,1.0,2010,"[7.070140384482051, 7.029404059604938, 6.73555...",2.688142,8.666364,9.608333,False
420,2009,2,0.00,0.00,0.35,0.55,0.10,0.00,0.0,0.0,0.0,0.0,1.0,2010,"[7.160324851430871, 7.879408381259646, 7.24311...",7.026918,8.299565,9.608333,False


### `confidence_intervals` method, and evaluating based on the REAL unemployment rates.
confidence_intervals method takes the samples created during sample_func2, and creates confidence intervals for each row in prunemp. It requires an input of sample_values, and does the following:

1. creates lower_ci: using np.percentile, we calculate the 2.5th percentile of all samples.
2. creates upper_ci: using np.percentile, we calculate the 97.5th percentile of all samples. Together we have 95% of all samples within these points.
3. returns a series where lower_ci is 'lower_ci' & upper_ci is 'upper_ci'.

Using this method on all rows 'sample_values' column, we create `ci_prunemp`, which takes all of the series created and turns them into a dataframe. Then, we combine ci_prunemp with pruncemp_short, using .join(), which adds the upper and lower confidence intervals for each row, to each row.


Then, it is important to add the actual unemployment rates from real_unemp into the prunemp_short dataframe. To do this:

1. I reset the prunemp_short indices, then re-set them after just for organization sake. If I didn't do this the index would just be reset after the merge.
2. I merged the real_unemp df with prunemp_short, on the forecast_year so that the unemployment rates are for the year that the predictions were made for.
3. I then dropped the YEAR1 and QUARTER1 to make the final df look more organized as well.


Finally, I created a column for prunemp_short titled `in_CI` which basically checks to see if the real unemployment rate for each row's target year is within each row's confidence interval. If it is within the bounds, the column shows True, and if its not, the column shows False.

In [311]:
percent_rationality = prunemp_short.groupby('ID').mean()['in_CI']
rational_forecasters_95 = (percent_rationality >=0.95).sum()/percent_rationality.shape
rational_forecasters_80 = (percent_rationality >=0.8).sum()/percent_rationality.shape
rational_forecasters_50 = (percent_rationality >=0.5).sum()/percent_rationality.shape
print(f'If rational forecasters are correct >= 95% of the time, the percent of rational forecasters is {rational_forecasters_95[0]*100:.2f}%')
print(f'If rational forecasters are correct >= 80% of the time, the percent of rational forecasters is {rational_forecasters_80[0]*100:.2f}%')
print(f'If rational forecasters are correct >= 50% of the time, the percent of rational forecasters is {rational_forecasters_50[0]*100:.2f}%')

If rational forecasters are correct >= 95% of the time, the percent of rational forecasters is 3.23%
If rational forecasters are correct >= 80% of the time, the percent of rational forecasters is 6.45%
If rational forecasters are correct >= 50% of the time, the percent of rational forecasters is 12.90%


### Evaluating Rational Forecasters

`percent_rationality` is created which makes a Series that takes all the prediction rows, sorted by ID, and takes the mean of their 'in_CI' column values. Since I used bools to evaluate whether the real unemployment rate was within the CIs, this means that each mean is the proportion of Trues, as True=1 and False=0. 

From there, I evaluated the overall rationality of forecasters using 3 different bounds. Specifically, I evaluated the percent of forecasters that had the actual unemployment rate within their CIs 50%, 80%, and 95% of the time, by taking the sum of forecasters in the percent_rationality series that were >= these percent values / the percent_rationality shape (total forecasters).

Based on the percent of rational forecasters listed above, it is safe to say that we can reject the null hypothesis listed in the slides. This means that these forecasters are NOT rational, and the observed unemployment rate is not drawn from forecaster distributions, since they are wrong more often than not. 